In [22]:
import pandas as pd
import numpy as np
import json

### Load and preprocess the data ###
df = pd.read_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/OT_vs_OK_combined_annotations.csv")

# Extract KEGG KOs and log2FoldChange
df = df[['KEGG_ko','baseMean', 'log2FoldChange','padj']]
df['KEGG_ko'] = df['KEGG_ko'].str.split(',')
df_exploded = df.explode('KEGG_ko')
df_exploded['KEGG_ko'] = df_exploded['KEGG_ko'].str.strip()
df_exploded = df_exploded.replace('-', np.nan).dropna()

with open("/home/nanopore/projects/rna_seq_workflow/resources/20250603_FCOM/enrichment_analysis/kegg_request/20250603_FCOM_kegg_cache.json", "r") as cache_file:
    kegg_cache = json.load(cache_file)
    
df_exploded["Pathways"] = df_exploded["KEGG_ko"].map(lambda ko: kegg_cache.get(ko, {}).get("pathways", []))
df_pathways = df_exploded.explode("Pathways").dropna(subset=["Pathways"])

In [25]:
def prepare_gsea_data(df, term_column, padj_floor=1e-300):
    df = df.copy()

    # Guard against padj == 0
    df["padj_safe"] = df["padj"].clip(lower=padj_floor)

    # Define combined weight
    df["weight"] = (
        df["baseMean"] * (-np.log10(df["padj_safe"]))
    )

    weighted_sum = (
        df.assign(weighted_lfc=df["log2FoldChange"] * df["weight"])
          .groupby("KEGG_ko")[["weighted_lfc", "weight"]]
          .sum()
    )
    
    ranked_list = (
        weighted_sum["weighted_lfc"] / weighted_sum["weight"]
    ).sort_values(ascending=False)
    
    gene_sets = (
        df.set_index(term_column)['KEGG_ko']
        .groupby(term_column)
        .apply(list)
        .to_dict()
    )
    return ranked_list, gene_sets

df_exploded

,KEGG_ko,baseMean,log2FoldChange,padj,Pathways
1,ko:K09831,4091.714326,5.927976,6.718029e-23,"[Steroid biosynthesis, Metabolic pathways, Bio..."
2,ko:K04563,728.521427,5.755917,1.434205e-21,"[MAPK signaling pathway - yeast, Cell cycle - ..."
2,ko:K05916,728.521427,5.755917,1.434205e-21,[]
3,ko:K10534,126.911183,5.535870,3.503635e-14,"[Nitrogen metabolism, Metabolic pathways, Micr..."
10,ko:K15877,6727.773797,3.669337,4.764562e-45,"[Nitrogen metabolism, Metabolic pathways, Micr..."
...,...,...,...,...,...
7662,ko:K20989,129.944426,-7.176103,5.863643e-52,[]
7663,ko:K16261,7337.423652,-7.319251,9.625535e-86,[]
7665,ko:K00463,311.914781,-8.125543,3.690634e-23,"[Tryptophan metabolism, Metabolic pathways, Bi..."
7667,ko:K16261,705.793728,-8.843190,2.394967e-137,[]


In [26]:
ranked_list_pathways, gene_sets_pathways = prepare_gsea_data(df_pathways, "Pathways")

ranked_list_pathways


KEGG_ko
ko:K09831    5.626879
ko:K15877    3.669337
ko:K00958    2.819685
ko:K04563    2.737133
ko:K00390    2.728562
               ...   
ko:K14541   -4.140986
ko:K00761   -4.346526
ko:K01181   -4.743456
ko:K00273   -4.920114
ko:K03787   -6.426687
Length: 2060, dtype: float64